# Lưu đầu vào hai pilot prompt — exact169 +161
**Accelerator None, Internet ON, không cần Add Input.** Chạy bằng tài khoản sở hữu hai Output; bật Secret `codex` (hoặc đổi SECRET_NAME). Tải cache169 và chỉ best.bin161, kiểm SHA mọi file, giữ nguyên binary; tạo **Dataset Private mới**. Không chạy CLIP/Torch/train. Chia sẻ Dataset cho Ngân sau khi xử lý xong để hai notebook gắn cùng input; không dùng token của bạn trong notebook của Ngân. Không ghi đè dataset/output đã tồn tại.


In [ ]:
from pathlib import Path
OWNER='ducanh2403'
SLUG='csa-prompt-pilot-inputs-v169'
TITLE='CSA prompt pilot assets v169 and v161'
SECRET_NAME='codex' # Tên Secret đã bật, token thuộc tài khoản ducanh2403.
DESTINATION=Path('/kaggle/working/csa_prompt_pilot_inputs_v169_dataset')


In [ ]:
import json,os,re,subprocess,sys,tempfile,shutil
from kaggle_secrets import UserSecretsClient
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError('Title/slug must have6–50characters')
if DESTINATION.exists():raise RuntimeError('Preserve existing output; choose fresh DESTINATION')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<4*1024**3:raise RuntimeError('Need4GiB free for download/copy; no files removed')
try:token=UserSecretsClient().get_secret(SECRET_NAME).strip()
except Exception:raise RuntimeError('Enable the named Secret for the owner of private outputs169/161') from None
if not token:raise RuntimeError('Empty API token')
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for k in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(k,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    process=subprocess.Popen(list(map(str,args)),env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace')
    for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
    if process.wait():raise RuntimeError('API command failed; preserve staged files and read redacted log above')
CLI=[sys.executable,'-c','from kaggle.cli import main; main()']
run_cli([sys.executable,'-m','pip','install','-q','kaggle==2.2.5'],authenticated=False)


In [ ]:
VERSION_PREFLIGHT="\nimport tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi=KaggleApi()\nfor version in ('169','161'):\n    with tempfile.TemporaryDirectory() as directory,patch.object(api,'build_kaggle_client') as factory:\n        endpoint=factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n        endpoint.return_value=SimpleNamespace(files=[],next_page_token=None,log=None)\n        api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/'+version,path=directory)\n        request=endpoint.call_args.args[0]\n        assert request.version_label=='v'+version and request.user_name=='ducanh2403' and request.kernel_slug=='2-vi-cai-kia-qua-nhieu','Exact version ignored'\nprint('Exact169/161 API routing preflight PASS; no network in this check.',flush=True)\n"
run_cli([sys.executable,'-u','-c',VERSION_PREFLIGHT])


In [ ]:
import hashlib
INVENTORY={'cache_version': 'ducanh2403/2-vi-cai-kia-qua-nhieu/169', 'checkpoint_version': 'ducanh2403/2-vi-cai-kia-qua-nhieu/161', 'cache_files': {'prompt_embeddings.npy': 'a6b77754db9f620b697d3e5d43c1cba784ccda477fdd98cbb707a2ddf9f38e56', 'prompt_texts.json': 'fbc6882106041babd46bda7ae7b6497eda1fab50fe64dc181f5970d474537f58', 'prompt_links.jsonl': '14e50c67fdc25b9247a8fb333bcf4733a360894330595207274d70672fb10ee9', 'cohort.json': 'da60b2a7b9c5304f3f3e1a1e61da8c713187d3f175c9671815a0bf63e4aac3e6', 'summary.json': '5cd88ae3a98bc7c18478135dc17b9677a90f01bde9fe5e792f8ef8f6e2d4d449', 'cache_manifest.json': '24c1c75fda9be90f6e1409aedc8d2bbecb04cb78e5570443a3f813b402b66486', 'cache_receipt.json': '3c3416d1d6d21ba693e4b5c9833d89b0479e89a712196453340fb9ad79d21a42', 'run_status.json': '8e2e790e898f91039c29519f3a10573c0e9f2bd54328625add986fa3e9fd2adb', 'source_receipt.json': '405bbf6582e171a55b82e37a22c9d4b3c9c8b2c0037e7942116ce42a73800e09', 'notebook.log': '9430b36cf0196988058959ce29792684d6dd394ae2b9d3465eb31314ad4ca7df'}, 'checkpoint_sha256': '58ce5dc0e30ac26add9bac8e24a2e9e03b91532bfdeece588997395bf9cfc16b'}
SOURCE_CODE='"""Byte-preserving CPU transport for exact169 cache and exact161 weights; no models."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nimport numpy as np\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream,\'sha256\').hexdigest()\n\n\ndef pattern(names):\n    return r\'(^|/)(\'+\'|\'.join(re.escape(n) for n in names)+r\')$\'\n\n\ndef locate(root,name,expected):\n    matches=[p for p in Path(root).rglob(name) if p.is_file() and sha(p)==expected]\n    if len(matches)!=1:\n        raise ValueError(f\'Need one exact {name}; found {len(matches)} matching SHA, no version fallback\')\n    return matches[0]\n\n\ndef package(cache_download,weight_download,destination,inventory):\n    destination=Path(destination)\n    if destination.exists():raise FileExistsError(\'Preserve existing destination; choose a fresh one\')\n    files=inventory[\'cache_files\']\n    if any(Path(n).name!=n or \'/\' in n or \'\\\\\' in n for n in files):raise ValueError(\'Unsafe payload name\')\n    receipt_path=locate(cache_download,\'cache_receipt.json\',files[\'cache_receipt.json\'])\n    root=receipt_path.parent\n    receipt=json.loads(receipt_path.read_text(encoding=\'utf-8\'))\n    if receipt[\'status\']!=\'PROMPT_TEXT_CACHE_COMPLETE\':raise ValueError(\'Incomplete cache\')\n    if any(n not in files or files[n]!=h for n,h in receipt[\'payload_sha256\'].items()):raise ValueError(\'Cache receipt identities differ\')\n    if set(receipt[\'payload_sha256\'])!={\'prompt_embeddings.npy\',\'prompt_texts.json\',\'prompt_links.jsonl\',\'cohort.json\',\'summary.json\',\'cache_manifest.json\'}:\n        raise ValueError(\'Cache receipt payload coverage differs\')\n    for n,h in files.items():\n        if not (root/n).is_file() or sha(root/n)!=h:raise ValueError(\'Missing/changed169 payload:\'+n)\n    state=json.loads((root/\'run_status.json\').read_text(encoding=\'utf-8\'))\n    if not state[\'complete\'] or state[\'training\'] or state[\'test_used\']:raise ValueError(\'Wrong cache run status\')\n    vectors=np.load(root/\'prompt_embeddings.npy\',allow_pickle=False)\n    if vectors.shape!=(318,512) or vectors.dtype!=np.float32 or not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors,axis=1),1,atol=2e-6,rtol=0):\n        raise ValueError(\'Embedding shape/dtype/norm differs\')\n    weight=locate(weight_download,\'best.bin\',inventory[\'checkpoint_sha256\'])\n    # Binary identity is checked without deserializing weights or importing Torch.\n    destination.mkdir(parents=True)\n    for n in files:shutil.copy2(root/n,destination/n)\n    shutil.copy2(weight,destination/\'csa_ce161_best.bin\')\n    expected={**files,\'csa_ce161_best.bin\':inventory[\'checkpoint_sha256\']}\n    for n,h in expected.items():\n        if sha(destination/n)!=h:raise ValueError(\'Copied bytes changed:\'+n)\n    report=dict(status=\'PROMPT_PILOT_INPUTS_PRESERVED\',cache_source=inventory[\'cache_version\'],\n        checkpoint_source=inventory[\'checkpoint_version\'],files_sha256=expected,\n        original_bytes=True,models_executed=False,training=False,test_used=False)\n    (destination/\'transport_receipt.json\').write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf-8\')\n    return report\n'
assert hashlib.sha256(SOURCE_CODE.encode()).hexdigest()=='bf168d11c53f64ff6ca348e620beb057aed93ea80f4205905df84c17fe73ba0a'
exec(compile(SOURCE_CODE,'package_prompt_inputs.py','exec'))


In [ ]:
DOWNLOAD=Path(tempfile.mkdtemp(prefix='csa-prompt-inputs-',dir='/tmp'))
CACHE_DOWNLOAD=DOWNLOAD/'cache169';CACHE_DOWNLOAD.mkdir()
WEIGHT_DOWNLOAD=DOWNLOAD/'weight161';WEIGHT_DOWNLOAD.mkdir()
run_cli(CLI+['kernels','output',INVENTORY['cache_version'],'-p',str(CACHE_DOWNLOAD),'--file-pattern',pattern(INVENTORY['cache_files'])])
run_cli(CLI+['kernels','output',INVENTORY['checkpoint_version'],'-p',str(WEIGHT_DOWNLOAD),'--file-pattern',pattern(['best.bin'])])
report=package(CACHE_DOWNLOAD,WEIGHT_DOWNLOAD,DESTINATION,INVENTORY)
print('Verified/copy unchanged:',len(report['files_sha256']),'files. No model loaded.',flush=True)


In [ ]:
CREATE_DATASET_SCRIPT="import sys\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi = KaggleApi()\napi.authenticate()\nresult = api.dataset_create_new(sys.argv[1], public=False, quiet=False,\n                                convert_to_csv=False, dir_mode='skip')\nif result is None or str(getattr(result, 'status', '')).lower() != 'ok' or getattr(result, 'error', None):\n    raise RuntimeError('Dataset creation rejected: ' + str(getattr(result, 'error', 'Missing success response')))\nprint('Dataset creation accepted by API:', result.url, flush=True)\n"

metadata_path=DESTINATION/'dataset-metadata.json'
if metadata_path.exists():raise RuntimeError('Preserve existing metadata; use fresh destination')
metadata=dict(title=TITLE,id=f'{OWNER}/{SLUG}',licenses=[dict(name='other')],
    description='Private research assets: original frozen CLIP prompt text cache from exact notebook169, mapping TRAIN10k and VALID5k only; original CSA SCST+CE best checkpoint from exact161. No TEST or images. Upstream data/model terms retained. No model executed during transport.')
try:
    metadata_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    run_cli([sys.executable,'-u','-c',CREATE_DATASET_SCRIPT,str(DESTINATION)])
finally:metadata_path.unlink(missing_ok=True)
print('API accepted creation; processing may still be active. Check page before retrying.',flush=True)
print(f'https://www.kaggle.com/datasets/{OWNER}/{SLUG}',flush=True)
print('After processing, grant Ngan access in Dataset sharing; do not give anyone your API token.',flush=True)


Sau khi Dataset xử lý xong, gửi path mount thực tế. Dự kiến `/kaggle/input/datasets/ducanh2403/csa-prompt-pilot-inputs-v169`. Dataset này gồm cache + một trọng số; **ROI134 và dataset COCO vẫn gắn riêng** khi train. Đây chỉ là bảo toàn đầu vào, không phải kết quả hai pilot.
